<a href="https://colab.research.google.com/github/YLysov0017/PETUXON/blob/master/ML_2026_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install tensorflow

In [2]:
from sklearn.datasets import load_iris
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

#### Установка персонального параметра генератора сл. чисел!!!

In [3]:
from tensorflow.keras.utils import set_random_seed

set_random_seed( 8 )

# для абсолютной повторяемости желательно еще включать детерминизм, особно при работе на GPU,
#но пока можно ограничиться только set_random_seed
#tf.config.experimental.enable_op_determinism()

#### Загрузка набора данных

In [ ]:
ds = pd.read_csv('/content/classify08.csv')

In [ ]:
x = ds['data']
y = ds['target']

#### предобработка данных:
* нормирование / стандартизация входов
* кодирование выходов
* разделение данных на обучающий и тестовые наборы

In [ ]:
xNorm = ( x - np.min(x, axis = 0) ) / ( np.max(x, axis = 0) - np.min(x, axis = 0) )

In [ ]:
ohe = OneHotEncoder( sparse_output = False )
yEnc = ohe.fit_transform( y.reshape(-1,1) )

In [ ]:
xTrain, xTest, yTrain, yTest = train_test_split( xNorm, yEnc, train_size = 0.67)

#### построение модели ИНС путем последовательного описания слоев в форме списка

In [ ]:
nn = Sequential( [
    Input( shape = ( xNorm.shape[1],) ),
    Dense( units = 8, activation = "relu" ),
    Dense( units = 4, activation = "relu" ),
    Dense( units = yEnc.shape[1], activation = "softmax" )
] )

nn.compile( loss = "categorical_crossentropy",
            optimizer = "adam", metrics = ["categorical_accuracy"] )

#### обучение модели на обучающем наборе с валидацией по 10% от обучающей выборки (последние 10% прмиеров)

In [ ]:
hist = nn.fit( xTrain, yTrain, epochs = 80, batch_size = 10, validation_split = 0.1 )

#### визуализация метрик

In [ ]:
plt.plot( hist.history["loss"], label = "Loss" )
plt.plot( hist.history["categorical_accuracy"], label = "Accuracy" )

plt.plot( hist.history["val_categorical_accuracy"], label = "Accuracy" )
plt.xlabel("Epochs")
plt.ylabel("Loss / Accuracy")
plt.legend()
plt.show()

#### оценка модели на тестовом наборе

In [ ]:
nn.evaluate(xTest, yTest)

#### использование модели для предсказания на произвольном примере (предоставление одного примера)

In [ ]:
# "произвольный" пример возьмем из тестового набора

sampleIndex = 6
yPred = nn.predict( xTest[ sampleIndex ].reshape(1,-1) ) # важно преобразовать 1 пример к массиву из 1 элемента (довавить 1 измерение)

predLabel = np.argmax( yPred )
actLabel = np.argmax( yTest[index] )
print(f"Предсказание: {predLabel}, На самом деле: {actLabel}")

In [ ]:
# посмтореть на сырой выход нейросети
yPred

### Классификация без One-Hot

Используем *xNorm* на входе и "родной" *y* с номерами меток на выходе

In [ ]:
x_train, x_test, y_train, y_test = train_test_split( xNorm, y, train_size = 0.67 )

Чтобы использовать метки, не прибегая к их бинарному кодированию, используем функцию потерь **sparse_categorical_crossentropy**

Однако на выходе нейросети должны по-прежнему быть три классифицирующих нейрона (sigmoid или softmax)

In [ ]:
nnSparse = Sequential( [
    Input( shape = ( xNorm.shape[1],) ),
    Dense( units = 12, activation = "relu" ),
    Dense( units = 8, activation = "relu" ),
    Dense( units = len(np.unique(y)), activation = "softmax" )
] )

nnSparse.compile( loss = "sparse_categorical_crossentropy",
            optimizer = "adam", metrics = ["sparse_categorical_accuracy"] )

In [ ]:
hist = nnSparse.fit( x_train, y_train, epochs = 100, batch_size = 10 )

In [ ]:
plt.plot( hist.history["loss"], label = "Loss" )
plt.plot( hist.history["sparse_categorical_accuracy"], label = "Accuracy" )
plt.xlabel("Epochs")
plt.ylabel("Loss / Accuracy")
plt.legend()
plt.show()

In [ ]:
nnSparse.evaluate( x_test, y_test )

### Работа с переобучением модели

Искусственно создаем переобученную модель и без изменения ее структуры (количество нейронов в слоях) добиваемся снижения эффекта переобучения:
* Слои прореживания Dropout
* L1, L2, L1L2 регуляризация